<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="http://mng.bz/orYv">Build a Large Language Model From Scratch</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/LLMs-from-scratch">https://github.com/rasbt/LLMs-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="http://mng.bz/orYv"><img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# Chapter 2: Working with Text Data
# 第 2 章：处理文本数据

Packages that are being used in this notebook:
本笔记本中使用的软件包：

In [1]:
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

torch version: 2.11.0+cpu
tiktoken version: 0.13.0


- This chapter covers data preparation and sampling to get input data "ready" for the LLM
- 本章涵盖了数据准备和采样，以使输入数据为 LLM 做好“准备”

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/01.webp?timestamp=1" width="500px">

## 2.1 Understanding word embeddings
## 2.1 理解词嵌入

- No code in this section
- 本节无代码

- There are many forms of embeddings; we focus on text embeddings in this book
- 嵌入有多种形式；本书我们专注于文本嵌入

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/02.webp" width="500px">

- LLMs work with embeddings in high-dimensional spaces (i.e., thousands of dimensions)
- LLM 在高维空间（即数千个维度）中处理嵌入

- Since we can't visualize such high-dimensional spaces (we humans think in 1, 2, or 3 dimensions), the figure below illustrates a 2-dimensional embedding space
- 由于我们无法可视化这种高维空间（我们人类以 1、2 或 3 维进行思考），下图展示了一个二维嵌入空间

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/03.webp" width="300px">

## 2.2 Tokenizing text
## 2.2 标记文本

- In this section, we tokenize text, which means breaking text into smaller units, such as individual words and punctuation characters
- 在本节中，我们将文本进行 Tokenize（标记化），这意味着将文本拆分为较小的单位，例如单个单词和标点符号

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/04.webp" width="300px">

- Load raw text we want to work with
- 加载我们想要处理的原始文本

- [The Verdict by Edith Wharton](https://en.wikisource.org/wiki/The_Verdict) is a public domain short story
- [Edith Wharton 的《The Verdict》](https://en.wikisource.org/wiki/The_Verdict) 是一篇公共领域短篇小说

In [2]:
import os
import requests

if not os.path.exists("the-verdict.txt"):
    url = (
        "https://raw.githubusercontent.com/rasbt/"
        "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
        "the-verdict.txt"
    )
    file_path = "the-verdict.txt"

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    with open(file_path, "wb") as f:
        f.write(response.content)


# The book originally used the following code below
# However, urllib uses older protocol settings that
# can cause problems for some readers using a VPN.
# The `requests` version above is more robust
# in that regard.

"""
import os
import urllib.request

if not os.path.exists("the-verdict.txt"):
    url = ("https://raw.githubusercontent.com/rasbt/"
           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"
           "the-verdict.txt")
    file_path = "the-verdict.txt"
    urllib.request.urlretrieve(url, file_path)
"""

'\nimport os\nimport urllib.request\n\nif not os.path.exists("the-verdict.txt"):\n    url = ("https://raw.githubusercontent.com/rasbt/"\n           "LLMs-from-scratch/main/ch02/01_main-chapter-code/"\n           "the-verdict.txt")\n    file_path = "the-verdict.txt"\n    urllib.request.urlretrieve(url, file_path)\n'

<br>

---

<br>

#### Troubleshooting SSL certificate errors
#### 排除 SSL 证书错误

- Some readers reported seeing ssl.SSLCertVerificationError: `SSL: CERTIFICATE_VERIFY_FAILED` when running `urllib.request.urlretrieve` in VSCode or Jupyter.
- 一些读者报告在 VSCode 或 Jupyter 中运行 `urllib.request.urlretrieve` 时看到 ssl.SSLCertVerificationError: `SSL: CERTIFICATE_VERIFY_FAILED`。

- This usually means Python's certificate bundle is outdated.
- 这通常意味着 Python 的证书包已过期。

**Fixes**
**修复方法**

- Use Python ≥ 3.9; you can check your Python version by executing the following code:
- 使用 Python ≥ 3.9；您可以通过执行以下代码来检查您的 Python 版本：

```python
import sys
print(sys.__version__)
```

- Upgrade the cert bundle:
- 升级证书包：

  - pip: `pip install --upgrade certifi`
  - pip: `pip install --upgrade certifi`

  - uv: `uv pip install --upgrade certifi`
  - uv: `uv pip install --upgrade certifi`

- Restart the Jupyter kernel after upgrading.
- 升级后重启 Jupyter kernel。

- If you still encounter an `ssl.SSLCertVerificationError` when executing the previous code cell, please see the discussion at [more information here on GitHub](https://github.com/rasbt/LLMs-from-scratch/pull/403)
- 如果您在执行前面的代码单元格时仍然遇到 `ssl.SSLCertVerificationError`，请参阅 [GitHub 上的更多信息](https://github.com/rasbt/LLMs-from-scratch/pull/403) 处的讨论

<br>

---

<br>

In [3]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total number of character:", len(raw_text))
print(raw_text[:99])

Total number of character: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


- The goal is to tokenize and embed this text for an LLM
- 目标是为 LLM 对此文本进行 Tokenize 和嵌入

- Let's develop a simple tokenizer based on some simple sample text that we can then later apply to the text above
- 让我们基于一些简单的示例文本开发一个简单的 Tokenizer，稍后我们可以将其应用于上面的文本

- The following regular expression will split on whitespaces
- 以下正则表达式将根据空格进行拆分

In [4]:
import re

text = "Hello, world. This, is a test."
# r'(\s)' 的含义：
# \s  : 匹配任何空白字符（空格、换行、制表符等）
# ()  : 捕获分组。在 re.split 中使用时，匹配到的分隔符（这里是空格）也会被保留在结果列表中
result = re.split(r'(\s)', text)

print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


- We don't only want to split on whitespaces but also commas and periods, so let's modify the regular expression to do that as well
- 我们不仅想根据空格拆分，还想根据逗号和句点拆分，所以让我们修改正则表达式也执行该操作

In [6]:
result = re.split(r'([,.]|\s)', text)

print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


- As we can see, this creates empty strings, let's remove them
- 正如我们所见，这会创建空字符串，让我们移除它们

In [7]:
# 使用列表推导式处理分词结果：
# item.strip() 用于移除字符串两端的空白字符（如空格、换行符）
# if item.strip() 用于检查处理后的字符串是否非空，从而过滤掉空字符串
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'This', ',', 'is', 'a', 'test', '.']


- This looks pretty good, but let's also handle other types of punctuation, such as periods, question marks, and so on
- 这看起来很不错，但让我们也处理其他类型的标点符号，如句点、问号等等

In [9]:
text = "Hello, world. Is this-- a test?"

# 正则表达式拆解说明：
# ( ... )          : 捕获分组，表示拆分出的分隔符也会保留在列表中
# [,. :;?_!"()\'] : 字符集，匹配其中列出的任何一个标点符号
# | --             : 或者匹配双破折号 "--"
# | \s             : 或者匹配空白字符（空格、换行等）
result = re.split(r'([,.:;?_!"()\']|--|\s)', text)

# 移除每个 token 前后的空格，并过滤掉空字符串
result = [item.strip() for item in result if item.strip()]
print(result)

['Hello', ',', 'world', '.', 'Is', 'this', '--', 'a', 'test', '?']


- This is pretty good, and we are now ready to apply this tokenization to the raw text
- 这非常好，我们现在准备好将此 Tokenization 应用于原始文本

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/05.webp" width="350px">

In [10]:
preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(preprocessed[:30])

['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


- Let's calculate the total number of tokens
- 让我们计算 Token 的总数

In [11]:
print(len(preprocessed))

4690


## 2.3 Converting tokens into token IDs
## 2.3 将 Token 转换为 Token ID

- Next, we convert the text tokens into token IDs that we can process via embedding layers later
- 接下来，我们将文本 Token 转换为 Token ID，以便稍后通过嵌入层进行处理

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/06.webp" width="500px">

- From these tokens, we can now build a vocabulary that consists of all the unique tokens
- 通过这些 Token，我们现在可以构建一个由所有唯一 Token 组成的 Vocabulary

In [12]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(vocab_size)

1130


In [13]:
vocab = {token:integer for integer,token in enumerate(all_words)}

- Below are the first 50 entries in this vocabulary:
- 以下是该 Vocabulary 中的前 50 个条目：

In [14]:
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 50:
        break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
('Burlington', 21)
('But', 22)
('By', 23)
('Carlo', 24)
('Chicago', 25)
('Claude', 26)
('Come', 27)
('Croft', 28)
('Destroyed', 29)
('Devonshire', 30)
('Don', 31)
('Dubarry', 32)
('Emperors', 33)
('Florence', 34)
('For', 35)
('Gallery', 36)
('Gideon', 37)
('Gisburn', 38)
('Gisburns', 39)
('Grafton', 40)
('Greek', 41)
('Grindle', 42)
('Grindles', 43)
('HAD', 44)
('Had', 45)
('Hang', 46)
('Has', 47)
('He', 48)
('Her', 49)
('Hermia', 50)


- Below, we illustrate the tokenization of a short sample text using a small vocabulary:
- 下面，我们展示了使用小型 Vocabulary 对短示例文本进行的 Tokenization：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/07.webp?123" width="500px">

- Putting it now all together into a tokenizer class
- 现在将这一切整合到一个 Tokenizer 类中

In [15]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    # 它通过匹配标点符号前面的空格，并利用 \1（反向引用）
    # 保留标点而删除空格，从而让解码后的文本更符合自然书写习惯。
    def decode(self, ids):
        # 将所有 token 用空格拼接成初步字符串
        text = " ".join([self.int_to_str[i] for i in ids])

        # 使用正则优化标点符号前后的空格：
        # \s+             : 匹配标点符号前的一个或多个空格
        # ([,.?!"()\'])   : 捕获组 1，匹配括号内的任意一个标点符号
        # r'\1'           : 替换内容。\1 代表引用捕获组 1 中的内容
        # 效果：将 "hello ," 替换为 "hello,"，即删除了标点前的多余空格
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

- The `encode` function turns text into token IDs
- `encode` 函数将文本转换为 Token ID

- The `decode` function turns token IDs back into text
- `decode` 函数将 Token ID 转回文本

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/08.webp?123" width="500px">

- We can use the tokenizer to encode (that is, tokenize) texts into integers
- 我们可以使用 Tokenizer 将文本 encode（即标记化）为整数

- These integers can then be embedded (later) as input of/for the LLM
- 这些整数随后可以（稍后）作为 LLM 的输入进行嵌入

In [16]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know,"
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


- We can decode the integers back into text
- 我们可以将整数 decode 回文本

In [17]:
tokenizer.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [18]:
tokenizer.decode(tokenizer.encode(text))

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

## 2.4 Adding special context tokens
## 2.4 添加特殊的上下文 Token

- It's useful to add some "special" tokens for unknown words and to denote the end of a text
- 为未知单词添加一些“特殊” Token 并表示文本结束是非常有用的

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/09.webp?123" width="500px">

- Some tokenizers use special tokens to help the LLM with additional context
- 一些 Tokenizer 使用特殊 Token 来帮助 LLM 获取额外的上下文

- Some of these special tokens are
- 其中一些特殊 Token 包括

- `[BOS]` (beginning of sequence) marks the beginning of text
- `[BOS]`（序列开始）标记文本的开始

- `[EOS]` (end of sequence) marks where the text ends (this is usually used to concatenate multiple unrelated texts, e.g., two different Wikipedia articles or two different books, and so on)
- `[EOS]`（序列结束）标记文本结束的地方（这通常用于连接多个不相关的文本，例如，两个不同的维基百科文章或两本不同的书等等）

- `[PAD]` (padding) if we train LLMs with a batch size greater than 1 (we may include multiple texts with different lengths; with the padding token we pad the shorter texts to the longest length so that all texts have an equal length)
- `[PAD]`（填充），如果我们在训练 LLM 时 batch size 大于 1（我们可能包含多个长度不同的文本；通过 padding token，我们将较短的文本填充到最长长度，以便所有文本都具有相同的长度）

- `[UNK]` to represent words that are not included in the vocabulary
- `[UNK]` 用来表示不包含在 Vocabulary 中的单词

- Note that GPT-2 does not need any of these tokens mentioned above but only uses an `<|endoftext|>` token to reduce complexity
- 请注意，GPT-2 不需要上面提到的任何这些 Token，而仅使用 `<|endoftext|>` Token 来降低复杂性

- The `<|endoftext|>` is analogous to the `[EOS]` token mentioned above
- `<|endoftext|>` 类似于上面提到的 `[EOS]` Token

- GPT also uses the `<|endoftext|>` for padding (since we typically use a mask when training on batched inputs, we would not attend padded tokens anyways, so it does not matter what these tokens are)
- GPT 还使用 `<|endoftext|>` 进行 padding（因为我们在对批量输入进行训练时通常使用掩码，所以无论如何我们都不会关注填充的 Token，因此这些 Token 是什么并不重要）

- GPT-2 does not use an `<UNK>` token for out-of-vocabulary words; instead, GPT-2 uses a byte-pair encoding (BPE) tokenizer, which breaks down words into subword units which we will discuss in a later section
- GPT-2 不对 Vocabulary 之外的单词使用 `<UNK>` Token；相反，GPT-2 使用 Byte-Pair Encoding (BPE) Tokenizer，它将单词拆分为子词单位，我们将在后面的章节中讨论

- We use the `<|endoftext|>` tokens between two independent sources of text:
- 我们在两个独立的文本源之间使用 `<|endoftext|>` Token：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/10.webp" width="500px">

- Let's see what happens if we tokenize the following text:
- 让我们看看如果我们对以下文本进行 Tokenize 会发生什么：

In [19]:
tokenizer = SimpleTokenizerV1(vocab)

text = "Hello, do you like tea. Is this-- a test?"

tokenizer.encode(text)

KeyError: 'Hello'

- The above produces an error because the word "Hello" is not contained in the vocabulary
- 上述操作产生错误，因为单词 "Hello" 不包含在 Vocabulary 中

- To deal with such cases, we can add special tokens like `"<|unk|>"` to the vocabulary to represent unknown words
- 为了处理此类情况，我们可以在 Vocabulary 中添加诸如 `"<|unk|>"` 之类的特殊 Token 来表示未知单词

- Since we are already extending the vocabulary, let's add another token called `"<|endoftext|>"` which is used in GPT-2 training to denote the end of a text (and it's also used between concatenated text, like if our training datasets consists of multiple articles, books, etc.)
- 既然我们已经在扩展 Vocabulary，让我们添加另一个名为 `"<|endoftext|>"` 的 Token，它在 GPT-2 训练中用于表示文本的结束（它也用于连接文本之间，例如，如果我们的训练数据集由多篇文章、书籍等组成）

In [20]:
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])

vocab = {token:integer for integer,token in enumerate(all_tokens)}

In [21]:
len(vocab.items())

1132

In [22]:
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


- We also need to adjust the tokenizer accordingly so that it knows when and how to use the new `<unk>` token
- 我们还需要相应地调整 Tokenizer，以便它知道何时以及如何使用新的 `<unk>` Token

In [23]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        # 主要就是修改了这里
        preprocessed = [
            item if item in self.str_to_int
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

Let's try to tokenize text with the modified tokenizer:
让我们尝试使用修改后的 Tokenizer 来标记文本：

In [24]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [25]:
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

In [26]:
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'

## 2.5 BytePair encoding
## 2.5 BytePair Encoding

- GPT-2 used BytePair encoding (BPE) as its tokenizer
- GPT-2 使用 BytePair Encoding (BPE) 作为其 Tokenizer

- it allows the model to break down words that aren't in its predefined vocabulary into smaller subword units or even individual characters, enabling it to handle out-of-vocabulary words
- 它允许模型将不在其预定义 Vocabulary 中的单词拆分为较小的子词单位甚至单个字符，从而使其能够处理 Vocabulary 之外的单词

- For instance, if GPT-2's vocabulary doesn't have the word "unfamiliarword," it might tokenize it as ["unfam", "iliar", "word"] or some other subword breakdown, depending on its trained BPE merges
- 例如，如果 GPT-2 的 Vocabulary 中没有 "unfamiliarword" 这个单词，它可能会将其标记为 ["unfam", "iliar", "word"] 或某些其他的子词分解，这取决于其训练的 BPE 合并

- The original BPE tokenizer can be found here: [https://github.com/openai/gpt-2/blob/master/src/encoder.py](https://github.com/openai/gpt-2/blob/master/src/encoder.py)
- 原始 BPE Tokenizer 可以在这里找到：[https://github.com/openai/gpt-2/blob/master/src/encoder.py](https://github.com/openai/gpt-2/blob/master/src/encoder.py)

- In this chapter, we are using the BPE tokenizer from OpenAI's open-source [tiktoken](https://github.com/openai/tiktoken) library, which implements its core algorithms in Rust to improve computational performance
- 在本章中，我们使用来自 OpenAI 开源 [tiktoken](https://github.com/openai/tiktoken) 库的 BPE Tokenizer，该库在 Rust 中实现其核心算法以提高计算性能

- I created a notebook in the [./bytepair_encoder](../02_bonus_bytepair-encoder) that compares these two implementations side-by-side (tiktoken was about 5x faster on the sample text)
- 我在 [./bytepair_encoder](../02_bonus_bytepair-encoder) 中创建了一个笔记本，并排比较了这两个实现（tiktoken 在示例文本上快了约 5 倍）

In [27]:
# pip install tiktoken

In [28]:
import importlib
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

tiktoken version: 0.13.0


In [29]:
tokenizer = tiktoken.get_encoding("gpt2")

In [30]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

# 使用 tiktoken 进行编码：
# allowed_special 参数用于显式指定允许处理的特殊 Token（如 GPT-2 的结束符）
# 如果不设置这个参数，文本中的 "<|endoftext|>" 可能会被当作普通文本处理或引发错误
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


In [31]:
strings = tokenizer.decode(integers)

print(strings)

Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.


- BPE tokenizers break down unknown words into subwords and individual characters:
- BPE Tokenizer 将未知单词分解为子词和单个字符：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/11.webp" width="300px">

## 2.6 Data sampling with a sliding window
## 2.6 使用滑动窗口进行数据采样

- We train LLMs to generate one word at a time, so we want to prepare the training data accordingly where the next word in a sequence represents the target to predict:
- 我们训练 LLM 每次生成一个单词，因此我们希望相应地准备训练数据，其中序列中的下一个单词代表要预测的目标：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/12.webp" width="400px">

In [32]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

5145


- For each text chunk, we want the inputs and targets
- 对于每个文本块，我们需要 Input 和 Target

- Since we want the model to predict the next word, the targets are the inputs shifted by one position to the right
- 由于我们希望模型预测下一个单词，因此 Target 是向右移动一个位置的 Input

In [35]:
enc_sample = enc_text[50:]

In [36]:
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print(f"x: {x}")
print(f"y:      {y}")

x: [290, 4920, 2241, 287]
y:      [4920, 2241, 287, 257]


- One by one, the prediction would look like as follows:
- 预测将逐一如下所示：

In [38]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)

[290] ----> 4920
[290, 4920] ----> 2241
[290, 4920, 2241] ----> 287
[290, 4920, 2241, 287] ----> 257


In [39]:
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

 and ---->  established
 and established ---->  himself
 and established himself ---->  in
 and established himself in ---->  a


- We will take care of the next-word prediction in a later chapter after we covered the attention mechanism
- 我们将在介绍了注意力机制后的后续章节中处理下个词预测

- For now, we implement a simple data loader that iterates over the input dataset and returns the inputs and targets shifted by one
- 目前，我们实现一个简单的数据加载器，它迭代输入数据集并返回偏移一位的 Input 和 Target

- Install and import PyTorch (see Appendix A for installation tips)
- 安装并导入 PyTorch（有关安装提示，请参阅附录 A）

In [40]:
import torch
print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cpu


- We use a sliding window approach, changing the position by +1:
- 我们使用滑动窗口方法，将位置改变 +1：

- Create dataset and dataloader that extract chunks from the input text dataset
- 创建从输入文本数据集中提取块的 Dataset 和 DataLoader

In [41]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    """
    此代码的目的：
    实现一个自定义的 PyTorch Dataset 类，用于将原始文本数据转换为 LLM 训练所需的输入（Input）和目标（Target）。
    它使用滑动窗口（Sliding Window）技术对文本进行分块，使得每个输入序列都有一个对应的、向右偏移一位的预测目标序列。
    """

    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # 对整个文本进行分词（Tokenize）
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})

        # 确保分词后的数量足够提取出至少一个 max_length 长度的块
        assert len(token_ids) > max_length, "分词后的输入长度必须至少等于 max_length+1"

        # 使用滑动窗口将分词后的文本切分为多个 max_length 的重叠序列
        for i in range(0, len(token_ids) - max_length, stride):
            # 提取输入块（当前窗口内的 token）
            input_chunk = token_ids[i:i + max_length]
            # 提取目标块（输入块向后偏移一位，即模型需要预测的下一个 token 序列）
            target_chunk = token_ids[i + 1: i + max_length + 1]

            # 将块转换为 PyTorch 张量并存储
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        """返回数据集中的样本总数"""
        return len(self.input_ids)

    def __getitem__(self, idx):
        """根据索引返回一对输入张量和目标张量"""
        return self.input_ids[idx], self.target_ids[idx]

In [42]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    """
    此代码的目的：
    该函数是一个工厂函数，用于便捷地创建一个 PyTorch DataLoader。
    它负责初始化分词器、构建 GPTDatasetV1 数据集实例，并配置加载器参数（如批大小、打乱、并行线程数等），
    以便在大模型训练过程中高效地提供数据批次。
    """

    # 初始化分词器（使用 GPT-2 的 BPE 编码）
    tokenizer = tiktoken.get_encoding("gpt2")

    # 创建自定义数据集实例
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # 创建数据加载器（DataLoader）
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

- Let's test the dataloader with a batch size of 1 for an LLM with a context size of 4:
- 让我们为 context size 为 4 的 LLM 测试 batch size 为 1 的 DataLoader：

In [43]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [44]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [45]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


- An example using stride equal to the context length (here: 4) as shown below:
- 使用步幅（stride）等于上下文长度（此处为 4）的示例如下所示：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/14.webp" width="500px">

- We can also create batched outputs
- 我们还可以创建批处理（batched）输出

- Note that we increase the stride here so that we don't have overlaps between the batches, since more overlap could lead to increased overfitting
- 请注意，我们在这里增加了步幅，以便批次之间没有重叠，因为过多的重叠可能会导致过拟合增加

In [46]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


## 2.7 Creating token embeddings
## 2.7 创建 Token 嵌入

- The data is already almost ready for an LLM
- 数据已经基本为 LLM 准备好了

- But lastly let us embed the tokens in a continuous vector representation using an embedding layer
- 但最后让我们使用嵌入层将 Token 嵌入到连续向量表示中

- Usually, these embedding layers are part of the LLM itself and are updated (trained) during model training
- 通常，这些嵌入层是 LLM 本身的一部分，并在模型训练期间更新（训练）

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/15.webp" width="400px">

- Suppose we have the following four input examples with input ids 2, 3, 5, and 1 (after tokenization):
- 假设我们有以下四个输入示例，输入 ID 为 2、3、5 和 1（标记化后）：

In [47]:
input_ids = torch.tensor([2, 3, 5, 1])

- For the sake of simplicity, suppose we have a small vocabulary of only 6 words and we want to create embeddings of size 3:
- 为简单起见，假设我们有一个仅包含 6 个单词的小型 Vocabulary，并且我们想要创建大小为 3 的嵌入：

In [48]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

- This would result in a 6x3 weight matrix:
- 这将产生一个 6x3 的权重矩阵：

In [49]:
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


- For those who are familiar with one-hot encoding, the embedding layer approach above is essentially just a more efficient way of implementing one-hot encoding followed by matrix multiplication in a fully-connected layer, which is described in the supplementary code in [./embedding_vs_matmul](../03_bonus_embedding-vs-matmul)
- 对于熟悉 One-hot Encoding 的人来说，上面的嵌入层方法本质上只是实现 One-hot Encoding 随后在全连接层中进行矩阵乘法的一种更有效的方式，这在 [./embedding_vs_matmul](../03_bonus_embedding-vs-matmul) 的补充代码中有所描述

- Because the embedding layer is just a more efficient implementation that is equivalent to the one-hot encoding and matrix-multiplication approach it can be seen as a neural network layer that can be optimized via backpropagation
- 因为嵌入层只是与 One-hot Encoding 和矩阵乘法方法等效的更有效的实现，它可以被视为可以通过反向传播优化的神经网络层

- To convert a token with id 3 into a 3-dimensional vector, we do the following:
- 要将 ID 为 3 的 Token 转换为 3 维向量，我们执行以下操作：

In [50]:
print(embedding_layer(torch.tensor([3])))

tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


- Note that the above is the 4th row in the `embedding_layer` weight matrix
- 请注意，上面是 `embedding_layer` 权重矩阵中的第 4 行

- To embed all four `input_ids` values above, we do
- 要嵌入上面的所有四个 `input_ids` 值，我们执行

In [51]:
print(embedding_layer(input_ids))

tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


- An embedding layer is essentially a look-up operation:
- 嵌入层本质上是一个查找（look-up）操作：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/16.webp?123" width="500px">

- **You may be interested in the bonus content comparing embedding layers with regular linear layers: [../03_bonus_embedding-vs-matmul](../03_bonus_embedding-vs-matmul)**
- **您可能对比较嵌入层与普通线性层的奖励内容感兴趣：[../03_bonus_embedding-vs-matmul](../03_bonus_embedding-vs-matmul)**

## 2.8 Encoding word positions
## 2.8 编码单词位置

- Embedding layer convert IDs into identical vector representations regardless of where they are located in the input sequence:
- 无论 ID 在输入序列中的位置如何，嵌入层都会将其转换为相同的向量表示：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/17.webp" width="400px">

- Positional embeddings are combined with the token embedding vector to form the input embeddings for a large language model:
- 位置嵌入（Positional Embeddings）与 Token 嵌入向量相结合，形成大语言模型的输入嵌入：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/18.webp" width="500px">

- The BytePair encoder has a vocabulary size of 50,257:
- BytePair Encoder 的 Vocabulary 大小为 50,257：

- Suppose we want to encode the input tokens into a 256-dimensional vector representation:
- 假设我们想要将输入 Token 编码为 256 维向量表示：

In [52]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

- If we sample data from the dataloader, we embed the tokens in each batch into a 256-dimensional vector
- 如果我们从 DataLoader 中采样数据，我们将每个 batch 中的 Token 嵌入到一个 256 维向量中

- If we have a batch size of 8 with 4 tokens each, this results in a 8 x 4 x 256 tensor:
- 如果我们的 batch size 为 8，每个 batch 有 4 个 Token，这将产生一个 8 x 4 x 256 的张量：

In [54]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [55]:
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [56]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
# print(token_embeddings)

torch.Size([8, 4, 256])


- GPT-2 uses absolute position embeddings, so we just create another embedding layer:
- GPT-2 使用绝对位置嵌入，所以我们只需创建另一个嵌入层：

In [57]:
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

# uncomment & execute the following line to see how the embedding layer weights look like
# print(pos_embedding_layer.weight)

### 关于“绝对位置嵌入”的深入理解

**理解误区：**
最初认为“绝对位置”是指给整本书或整个数据集的所有 Token 统一标上唯一的序号（例如从 1 排到 10 万），然后根据这个全局序号产出 Embedding。

**正确理解：**
1. **窗口内的绝对性**：在 LLM 中，所谓的“绝对”是指在模型当前的**上下文窗口（Context Window）**内的位置是固定的。无论输入的是哪一段文本，窗口内的第 1 个位置永远对应索引 `0`，第 2 个位置永远对应索引 `1`。
2. **实现方式**：由于模型只能处理固定长度（如 `max_length=4`）的序列，因此位置嵌入矩阵只需要定义这么多行。每个 Batch 的数据进入模型时，都会重新从索引 `0` 开始映射位置向量。
3. **黑盒视角**：可以将位置嵌入看作是给窗口内的每个“座位”配发的固定编号。它告诉模型每个词在当前这一小段话里的相对顺序，而不需要关心这个词在原始巨长文本里的全局编号。

In [62]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
# print(pos_embeddings)

torch.Size([4, 256])


- To create the input embeddings used in an LLM, we simply add the token and the positional embeddings:
- 为了创建 LLM 中使用的输入嵌入，我们只需将 Token 嵌入和位置嵌入相加：

In [63]:
input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)

# uncomment & execute the following line to see how the embeddings look like
# print(input_embeddings)

torch.Size([8, 4, 256])


- In the initial phase of the input processing workflow, the input text is segmented into separate tokens
- 在输入处理工作流程的初始阶段，输入文本被分割成单独的 Token

- Following this segmentation, these tokens are transformed into token IDs based on a predefined vocabulary:
- 在此分割之后，这些 Token 根据预定义的 Vocabulary 转换为 Token ID：

<img src="https://sebastianraschka.com/images/LLMs-from-scratch-images/ch02_compressed/19.webp" width="400px">

## Summary and takeaways
## 总结与要点

See the [./dataloader.ipynb](./dataloader.ipynb) code notebook, which is a concise version of the data loader that we implemented in this chapter and will need for training the GPT model in upcoming chapters.
请参阅 [./dataloader.ipynb](./dataloader.ipynb) 代码笔记本，这是我们在本章中实现的 DataLoader 的简洁版本，在接下来的章节中训练 GPT 模型时将需要用到它。

See [./exercise-solutions.ipynb](./exercise-solutions.ipynb) for the exercise solutions.
有关练习解答，请参阅 [./exercise-solutions.ipynb](./exercise-solutions.ipynb)。

See the [Byte Pair Encoding (BPE) Tokenizer From Scratch](../02_bonus_bytepair-encoder/compare-bpe-tiktoken.ipynb) notebook if you are interested in learning how the GPT-2 tokenizer can be implemented and trained from scratch.
如果您有兴趣了解如何从头开始实现和训练 GPT-2 Tokenizer，请参阅 [从头开始的 Byte Pair Encoding (BPE) Tokenizer](../02_bonus_bytepair-encoder/compare-bpe-tiktoken.ipynb) 笔记本。